# PlanScope — full-scale training on a GPU (Colab)

The committed model was trained on a CPU laptop at reduced scale (see README → Results). This notebook runs the
same code at full scale: 3,000 synthetic sheets, 1024-px tiles, 60 epochs. Runtime → Change runtime type → GPU.
Expect ~1 h for data generation and ~2–4 h of training on a T4.

In [ ]:
!git clone https://github.com/Akhil-Nistala/planscope.git
%cd planscope
!apt-get -qq install -y fonts-noto-cjk > /dev/null   # Japanese fonts for the generator
!pip -q install pymupdf==2.1 PyYAML==6.0.3 jsonschema==4.26.0 opencv-python-headless pillow pytest
!pip -q install --no-deps ultralytics==8.4.202
!pip -q install matplotlib polars psutil requests ultralytics-thop

In [ ]:
# 3,000 sheets (~1.8 GB); the test split keeps its PDFs for end-to-end evaluation
!python -m data_gen.generate --out data/synth --n 3000 --workers 8
!python -m training.tile_dataset --src data/synth --out data/synth_tiles --tile 1024 --max-tiles 0

In [ ]:
# tile model (used by the pipeline) and the full-sheet baseline (for the ablation)
!python -m training.train --data data/synth_tiles/dataset.yaml --name tiles_1024 --imgsz 1024 --epochs 60 --batch 16 --device 0 --workers 8
!python -m training.train --data data/synth/dataset.yaml --name full_1024 --imgsz 1024 --epochs 60 --batch 16 --device 0 --workers 8

In [ ]:
# end-to-end evaluation on the synthetic test split (set detector.tiling.tile: 1024 for the 1024 model)
!python -m eval.evaluate --data data/synth --split test --weights models/tiles_1024.pt --tile 1024 --name tiles_1024
!python -m eval.evaluate --data data/synth --split test --weights models/full_1024.pt --no-tiling --name full_1024

In [ ]:
from google.colab import files
files.download('models/tiles_1024.pt'); files.download('models/tiles_1024.json')